In [1]:
#try ML OS on one realization

In [7]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

import numpy as np
from matplotlib import pyplot as plt
import discovery as ds

from la_forge.core import Core

import defiant
from defiant import OptimalStatistic
from defiant import utils, orf_functions
from defiant import plotting as defplot

from defiant.extra import mdc1_utils
import glob


print("Using defiant from:",defiant.__file__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using defiant from: /Users/ashokan/osu/sarahspta/defiant/defiant/__init__.py


In [5]:
from defiant.extra.fun import what_kind_of_OS_are_you

what_kind_of_OS_are_you()


______ ___________ _____  ___   _   _ _____ 
|  _  \  ___|  ___|_   _|/ _ \ | \ | |_   _|
| | | | |__ | |_    | | / /_\ \|  \| | | |  
| | | |  __||  _|   | | |  _  || . ` | | |  
| |/ /| |___| |    _| |_| | | || |\  | | |  
|___/ \____/\_|    \___/\_| |_/\_| \_/ \_/                                          
 



Welcome to the next hottest gameshow in NANOGrav: What kind of OS are you? 

In this game, we will determine what kind of Optimal Statistic you are based
on how you answer the following questions! 


Here we go!
Question 1: How many Overlap Reduction Functions (ORFs) are you using?
In other words, how many correlation patters are you searching for? [HD, Dipole, etc.]
Single ORF means Single-Component!


Question 2: Would you estimate that your PTA Gravitational Wave Background (GWB)
signal is in the weak-signal (1), intermediate-signal (2), or strong-signal (3) regimes?
Type the number corresponding to your answer or 0 if you do not know.
Weak signal regime. Pair covariance i

In [ ]:
data = '..//NG20v1p1SimData/data/NG20v1p1wGWB/realization_0/'
feathers = sorted(glob.glob(data+'*.feather'))
psrs = []
for feather in feathers:
    psrs.append(ds.Pulsar.read_feather(feather))

In [17]:
# find the maximum time span to set GW frequency sampling
tmin = [p.toas.min() for p in psrs]
tmax = [p.toas.max() for p in psrs]
Tspan = np.max(tmax) - np.min(tmin)


In [19]:
from enterprise.signals import parameter, gp_signals, white_signals, signal_base
import enterprise.signals
# timing model
tm = gp_signals.TimingModel()

# white noise
ef = white_signals.MeasurementNoise(efac=1, log10_t2equad=1)

# red noise (powerlaw with 30 frequencies)
pl = enterprise.signals.utils.powerlaw(log10_A=1, gamma=3)
rn = gp_signals.FourierBasisGP(spectrum=pl, components=30, Tspan=Tspan)

# gwb
# We pass this signal the power-law spectrum as well as the standard
# Hellings and Downs ORF
orf = enterprise.signals.utils.hd_orf()
crn = gp_signals.FourierBasisCommonGP(pl, orf, components=30, name='gw', Tspan=Tspan)


s = ef + rn + tm  + crn 
pta = signal_base.PTA([s(p) for p in psrs])

In [20]:
OS_obj = OptimalStatistic(psrs,pta=pta,gwb_name='gw')

In [28]:
OS_obj.set_orf(orfs=['hd'])
OS_obj.compute_OS()

AttributeError: 'NoneType' object has no attribute 'keys'

In [25]:
OS_obj.compute_OS??

Signature:
OS_obj.compute_OS(
    params=None,
    N=1,
    gamma=None,
    pair_covariance=False,
    return_pair_vals=True,
    fisher_diag=False,
    use_tqdm=True,
)
Source:   
    def compute_OS(self, params=None, N=1, gamma=None, pair_covariance=False, 
                   return_pair_vals=True, fisher_diag=False, use_tqdm=True):
        """Compute the OS and its various modifications.

        This is one of 2 main methods of the `OptimalStatistic` class. This method
        can compute any flavor of the OS which uses broadband estimation (i.e. constructs
        a single estimator for the whole spectrum). There are many forms in which you can
        use this method, and checking the decision tree is best for determining exactly
        what you might want and what parameters to set to accomplish that. 
        NOTE: Since this method's outputs can vary widely, it will return in a dictionary
        with the keys being the output names.

        The basic usage of this method ca